# 03 — Fine-tune WangchanBERTa (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Riewta/mlproj/blob/main/notebooks/03_wangchanberta.ipynb)

fine-tune [`airesearch/wangchanberta-base-att-spm-uncased`](https://huggingface.co/airesearch/wangchanberta-base-att-spm-uncased)
บนงานเดียวกับ baseline แล้วเทียบกันแบบยุติธรรม: **split เดียวกัน** (`src/data/splits.py`), **metric เดียวกัน** (`src/metrics.py`)
และวัด latency บน **เครื่องเดียวกัน**

**ก่อนรัน:** Runtime → Change runtime type → **T4 GPU** (ฟรี)

**วิธีใช้ notebook นี้**
1. แก้ `CONFIG` แล้วรันทั้ง notebook หนึ่งครั้ง = หนึ่ง experiment (ทุก run ถูกบันทึกใน MLflow)
2. ลองหลาย config แล้วเลือกตัวที่ดีที่สุด **จาก val เท่านั้น**
3. ตั้ง `RUN_TEST = True` แล้วรัน config ที่เลือก **ครั้งเดียว** เพื่อได้ตัวเลข test (กติกาเดียวกับ 02)

**ทำไม transformer น่าจะดีกว่า TF-IDF:** TF-IDF มองรีวิวเป็นถุงคำ ไม่รู้ลำดับ
ส่วน WangchanBERTa ถูก pretrain จากข้อความไทยหลายสิบ GB จึงรู้ความหมายตามบริบท
จุดที่ baseline พลาดใน 02 คือ **รีวิวผสม** ที่ชมก่อนแล้วติทีหลัง และ **"ไม่" ที่ไม่ได้แปลว่าลบ** ("ไม่มีเวลา") ซึ่งเป็นเรื่องของบริบทล้วนๆ

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("mlproj").exists():
        !git clone -q https://github.com/Riewta/mlproj
    %cd mlproj
    !pip install -q -r requirements-colab.txt
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/mlproj")   # keep runs and models after the session ends
else:
    OUT = Path.cwd().parent / "outputs" if Path.cwd().name == "notebooks" else Path.cwd() / "outputs"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## 1. Config

| ตัวแปร | ความหมาย | เหตุผลของค่าเริ่มต้น |
|---|---|---|
| `max_length` | จำนวน token สูงสุด | 416 = ความยาวที่ใช้ตอน pretrain (จาก 01: ตัดราว 3–10% ของรีวิว) |
| `truncation` | ตัดรีวิวยาวยังไง | `head_tail` เก็บต้น 128 + ท้าย เพราะรีวิวมักสรุปความรู้สึกไว้ท้าย ลองเทียบกับ `head` |
| `lr` | learning rate | 3e-5 เป็นช่วงมาตรฐานของ BERT-base (2e-5 ถึง 5e-5) สูงไปจะลืมสิ่งที่ pretrain มา |
| `epochs` | จำนวนรอบ | 3 พอสำหรับข้อมูล 25k; เลือก epoch ที่ val ดีที่สุดอัตโนมัติ |
| `batch_size` | ขนาด batch | 16 พอดีกับ T4 16 GB ที่ 416 token + fp16 |

**experiment ที่แนะนำ** (ทีละ run): `head_tail` 416 → `head` 416 → ถ้าเวลาไม่พอ ลอง `head_tail` 256 (เร็วกว่ามาก)

In [ ]:
CONFIG = dict(
    max_length=416,
    truncation="head_tail",   # "head" or "head_tail"
    head=128,
    lr=3e-5,
    epochs=3,
    batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    seed=42,
)
RUN_TEST = False   # True only for the config you picked on val

SMOKE = os.environ.get("SMOKE") == "1"   # tiny run to check the code works (no GPU needed)
if SMOKE:
    CONFIG.update(max_length=64, epochs=1, batch_size=8)
    RUN_TEST = True
RUN_NAME = f"{CONFIG['truncation']}-{CONFIG['max_length']}-lr{CONFIG['lr']}" + ("-smoke" if SMOKE else "")
RUN_NAME

In [ ]:
import json, time
import numpy as np
import pandas as pd
import torch
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)

from src.data.splits import wongnai_splits, wisesight_test
from src.models.transformer import MODEL_NAME, encode
from src.metrics import sentiment_metrics, bootstrap_ci

set_seed(CONFIG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 2. ข้อมูลและ tokenization

`encode()` ใน `src/models/transformer.py` ทำสามอย่างที่ต้องตรงกับตอน pretrain:
- **แปลงเป็นตัวพิมพ์เล็ก:** โมเดลเป็น uncased แต่ tokenizer ไม่แปลงให้ ตัวพิมพ์ใหญ่จะกลายเป็น `<unk>` ("MK" → `<unk>`)
- **เว้นวรรคเป็น `<_>`:** ตอน pretrain เว้นวรรคถูกแทนด้วย token นี้
- **ตัดความยาวเอง:** tokenizer ตัวนี้ไม่มีค่าความยาวสูงสุด ถ้าไม่ตัด รีวิวยาวจะเกิน 512 ตำแหน่งแล้ว error

ทั้งหมดอยู่ใน `src/` เพราะ API ใน Phase 4 ต้องเตรียม input แบบเดียวกันทุกตัวอักษร

In [ ]:
data = wongnai_splits()
data["wisesight"] = wisesight_test()
if SMOKE:
    data = {k: pd.concat([g.head(40) for _, g in d.groupby("label")]).reset_index(drop=True) for k, d in data.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = {k: encode(tokenizer, d.text, CONFIG["max_length"], CONFIG["truncation"], CONFIG["head"]) for k, d in data.items()}

full_len = pd.Series([len(x) for x in tokenizer([t.lower() for t in data["train"].text], add_special_tokens=False)["input_ids"]])
print(f"train reviews longer than {CONFIG['max_length']} tokens (truncated): {(full_len > CONFIG['max_length'] - 2).mean():.1%}")
print({k: len(v) for k, v in enc.items()})

In [ ]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, ids, labels):
        self.ids, self.labels = ids, list(labels)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return {"input_ids": self.ids[i], "labels": int(self.labels[i])}

ds = {k: ReviewDataset(enc[k], data[k].label) for k in data}

## 3. เทรน

- `Trainer` ประเมิน val ทุก epoch แล้ว **เก็บ checkpoint ที่ macro-F1 บน val ดีที่สุด** (`load_best_model_at_end`)
  ไม่ใช่ epoch สุดท้าย เพราะ fine-tune นานไปอาจ overfit
- `fp16` บน GPU: คำนวณด้วยความละเอียดครึ่งเดียว เร็วขึ้นราว 2 เท่าและใช้ memory น้อยลง แทบไม่เสียความแม่นยำ
- `DataCollatorWithPadding` เติม padding ทีละ batch ตามรีวิวที่ยาวที่สุดใน batch นั้น แทนที่จะเติมทุกรีวิวให้ยาว 416 ทำให้เร็วขึ้นมาก
- ทุก run บันทึกลง **MLflow** (ไฟล์ `mlflow.db` ใน Google Drive) ทั้ง config และกราฟ loss/F1 ดูย้อนหลังด้วย `mlflow ui` (ดูท้าย notebook)

In [ ]:
import shutil
import mlflow

# SQLite on Colab's local disk while training (SQLite on Google Drive can hit lock errors),
# copied to Drive after the run so every run accumulates in one file.
DB = Path("/content/mlflow.db") if IN_COLAB else OUT / "mlflow.db"
if IN_COLAB and (OUT / "mlflow.db").exists():
    shutil.copy(OUT / "mlflow.db", DB)
mlflow.set_tracking_uri(f"sqlite:///{DB}")
mlflow.set_experiment("wangchanberta-sentiment")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return sentiment_metrics(labels, logits.argmax(-1))

# warmup as an integer step count: works on both transformers 4.x and 5.x
steps_per_epoch = -(-len(ds["train"]) // CONFIG["batch_size"])
warmup_steps = int(steps_per_epoch * CONFIG["epochs"] * CONFIG["warmup_ratio"])

args = TrainingArguments(
    output_dir=str(OUT / "checkpoints" / RUN_NAME),
    learning_rate=CONFIG["lr"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=64,
    warmup_steps=warmup_steps,
    weight_decay=CONFIG["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=DEVICE == "cuda",
    use_cpu=DEVICE == "cpu",
    logging_steps=50,
    report_to=["mlflow"],
    run_name=RUN_NAME,
    seed=CONFIG["seed"],
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params({f"cfg_{k}": v for k, v in CONFIG.items()})
    start = time.time()
    trainer.train()
    val = trainer.evaluate()
    mlflow.log_metric("train_minutes", (time.time() - start) / 60)
if IN_COLAB:
    shutil.copy(DB, OUT / "mlflow.db")

## 4. เทียบบน val

baseline (TF-IDF + LR) ได้ macro-F1 บน val = **0.839** (จาก 02) ถ้า run นี้ไม่ชนะชัดเจน ก็ไม่คุ้มกับความช้าและขนาดที่เพิ่มขึ้น

In [ ]:
def predict_proba(name):
    logits = trainer.predict(ds[name]).predictions
    return torch.softmax(torch.tensor(logits).float(), -1)[:, 1].numpy()

p = {"val": predict_proba("val")}
val_metrics = sentiment_metrics(data["val"].label, (p["val"] >= 0.5).astype(int))
pd.DataFrame({"baseline (02)": {"macro_f1": 0.839, "f1_negative": 0.701}, RUN_NAME: val_metrics}).T.round(3)

**สิ่งที่ควรดู** (ยังไม่มีผล ต้องรันก่อน)
- macro-F1 บน val ชนะ 0.839 กี่จุด ถ้าชนะไม่ถึง ~0.02 อาจเป็นแค่ noise (val มีรีวิวลบแค่ ~225 รีวิว)
- F1 ของ class ลบ (baseline 0.701) คือตัวที่สำคัญที่สุด เพราะเป็นจุดอ่อนของ baseline
- ใน MLflow: val loss เริ่มขึ้นตั้งแต่ epoch ไหน ถ้า epoch 1 ดีที่สุด แปลว่าโมเดลเรียนเร็วและ overfit ไว

## 5. Test (รันครั้งเดียว สำหรับ config ที่เลือกแล้ว)

ตั้ง `RUN_TEST = True` เฉพาะตอนรัน config สุดท้าย เซลล์ข้างล่างจะข้ามไปถ้ายังเป็น `False`

In [ ]:
baseline = json.loads((ROOT / "reports" / "metrics" / "baseline.json").read_text())

if RUN_TEST:
    rows, cis = {}, {}
    for name, key in [("test", "wongnai_test"), ("wisesight", "wisesight_test")]:
        p[name] = predict_proba(name)
        pred = (p[name] >= 0.5).astype(int)
        rows[f"baseline | {name}"] = baseline[key]
        rows[f"wangchanberta | {name}"] = sentiment_metrics(data[name].label, pred)
        cis[name] = bootstrap_ci(data[name].label, pred)
        print(f"{name:10s} macro-F1 95% CI: [{cis[name][0]:.3f}, {cis[name][1]:.3f}]")
    display(pd.DataFrame(rows).T.round(3))

**อ่านผลยังไง:** baseline บน Wongnai test ได้ 0.862 (CI 0.839–0.884)
ถ้า CI ของ WangchanBERTa **ไม่ซ้อน** กับช่วงนี้ ถึงจะพูดได้เต็มปากว่าดีกว่า
Wisesight (baseline 0.473) คือจุดที่คาดว่าจะเห็นความต่างมากที่สุด

## 6. Latency และขนาด (เครื่องเดียวกัน)

latency ของ baseline ใน 02 วัดบน Mac ส่วนตรงนี้วัดบน Colab ถ้าเอาตัวเลขต่างเครื่องมาเทียบกันจะไม่ยุติธรรม
จึง **เทรน baseline ใหม่ตรงนี้** (ใช้เวลาราว 1–2 นาที) แล้ววัดทั้งสองโมเดลบน CPU เครื่องเดียวกัน
CPU สำคัญเพราะ Hugging Face Spaces แบบฟรี (ที่จะ deploy ใน Phase 4) **ไม่มี GPU**

In [ ]:
from src.cleaner import ThaiTextCleaner
from src.models.baseline import build_baseline

cleaner = ThaiTextCleaner()
base = build_baseline(C=50, class_weight=None).fit(
    [" ".join(cleaner.clean(t)) for t in data["train"].text], data["train"].label)

def latency_ms(fn, texts):
    times = []
    for t in texts:
        start = time.perf_counter(); fn(t); times.append((time.perf_counter() - start) * 1000)
    return float(np.percentile(times, 50))

cpu_model = trainer.model.float().to("cpu").eval()

@torch.no_grad()
def bert_predict(text, model=cpu_model, device="cpu"):
    ids = torch.tensor([encode(tokenizer, [text], CONFIG["max_length"], CONFIG["truncation"], CONFIG["head"])[0]], device=device)
    return model(input_ids=ids).logits.softmax(-1)

sample = data["test"].text.sample(min(100, len(data["test"])), random_state=0).tolist()
lat = {
    "baseline (CPU)": latency_ms(lambda t: base.predict_proba([" ".join(cleaner.clean(t))]), sample),
    "wangchanberta (CPU)": latency_ms(bert_predict, sample),
}
if DEVICE == "cuda":
    gpu_model = trainer.model.to("cuda")
    lat["wangchanberta (GPU)"] = latency_ms(lambda t: bert_predict(t, gpu_model, "cuda"), sample)

size_mb = sum(p_.numel() * 4 for p_ in cpu_model.parameters()) / 1e6   # fp32 weights
print({k: f"{v:.1f} ms" for k, v in lat.items()}, f"| model size ≈ {size_mb:.0f} MB (fp32)")

## 7. บันทึกผล

- `reports/metrics/wangchanberta.json` → ใช้เติมแถวที่สองในตาราง README
- `reports/predictions/*.csv` → ความน่าจะเป็นต่อรีวิว (เรียงตาม split ใน `src/data/splits.py`) ให้ `04_error_analysis` ใช้ได้โดยไม่ต้องมี GPU
- โมเดลเต็มเก็บใน Google Drive (ใหญ่เกินกว่าจะใส่ git)

บน Colab ไฟล์ใน `/content/mlproj` จะหายเมื่อปิด session จึงคัดลอกไฟล์ผลไปไว้ใน Drive ด้วย
ดาวน์โหลดจาก Drive แล้ว commit เข้า repo จากเครื่องตัวเอง

In [ ]:
if RUN_TEST:
    for folder in ["metrics", "predictions"]:
        (ROOT / "reports" / folder).mkdir(parents=True, exist_ok=True)

    summary = {
        "model": MODEL_NAME,
        "params": {**CONFIG, "threshold": 0.5},
        "wongnai_test": sentiment_metrics(data["test"].label, (p["test"] >= 0.5).astype(int)),
        "wongnai_test_ci": cis["test"],
        "wisesight_test": sentiment_metrics(data["wisesight"].label, (p["wisesight"] >= 0.5).astype(int)),
        "latency_ms_p50_cpu": round(lat["wangchanberta (CPU)"], 1),
        "baseline_latency_ms_p50_cpu_same_machine": round(lat["baseline (CPU)"], 1),
        "size_mb": round(size_mb),
        "smoke_test": SMOKE,
    }
    files = {ROOT / "reports" / "metrics" / f"wangchanberta{'_smoke' if SMOKE else ''}.json": json.dumps(summary, indent=2)}
    for name in ["val", "test", "wisesight"]:
        files[ROOT / "reports" / "predictions" / f"wangchanberta_{name}{'_smoke' if SMOKE else ''}.csv"] = \
            pd.DataFrame({"label": data[name].label.values, "p_positive": p[name].round(4)}).to_csv(index=False)
    for path, text in files.items():
        path.write_text(text)
        if IN_COLAB:
            (OUT / path.name).write_text(text)

    trainer.save_model(str(OUT / "model" / RUN_NAME))
    tokenizer.save_pretrained(str(OUT / "model" / RUN_NAME))
    print("saved:", *files, OUT / "model" / RUN_NAME, sep="\n  ")

## 8. ดูผลทุก run ใน MLflow

ดาวน์โหลดไฟล์ `MyDrive/mlproj/mlflow.db` มาที่เครื่อง แล้วรันในโฟลเดอร์เดียวกัน:

```bash
pip install mlflow
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

เปิด http://localhost:5000 จะเห็นทุก run เทียบกันได้ทั้ง config, val macro-F1 และกราฟ loss
(ถ้าอยากได้ dashboard ออนไลน์ที่แชร์ link ได้ เปลี่ยน `report_to=["mlflow"]` เป็น `["wandb"]` แล้ว login Weights & Biases)

**ถัดไป:** `04_error_analysis` อ่านไฟล์ predictions ของทั้งสองโมเดล แล้วดูว่า WangchanBERTa แก้เคสไหนได้ และยังพลาดเคสไหน
(ประชด, รีวิวผสม, "ไม่" ที่ไม่ได้แปลว่าลบ)